In [12]:
# If needed once in this environment:
# %pip install python-dotenv sqlalchemy pyodbc pandas numpy scikit-learn lifelines

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    from lifelines import CoxPHFitter
except ImportError:
    CoxPHFitter = None

sys.path.insert(0, '..')

from util.db import create_db_engine
from util.sql_loader import read_sql_file
from util.sql_runner import execute_sql_and_read_temp_table
from util.xpd_features import build_xpd_features_v6, build_loan_level_payin_v6

In [2]:
# Paths
project_root = Path.cwd().resolve().parent  # .../yieldCurve_augmenting
env_path = project_root / '.env'
sql_path = project_root / 'sql_scripts' / 'jcx_raw_harvey_v6.sql'

# Load SQL and execute temp-table workflow
engine = create_db_engine(env_path)
setup_sql = read_sql_file(sql_path)

raw_df = execute_sql_and_read_temp_table(engine, setup_sql, '#t17_final')

print('SQL loaded from:', sql_path)
print('Using env file:', env_path)
print('raw_df shape:', raw_df.shape)
raw_df.head()

SQL loaded from: /Users/starsrain/2025_concord/yieldCurve_augmenting/sql_scripts/jcx_raw_harvey_v6.sql
Using env file: /Users/starsrain/2025_concord/yieldCurve_augmenting/.env
raw_df shape: (862101, 29)


,LoanID,AppYear,AppMonth,AppWeek,CustType,Frequency,Application_ID,PortFolioID,LoanStatus,NewlyScored,...,LoanPaidOff,isLoanDefault,isRecentInstall,isInstallDefault,ThirdPartyCollected,PartialCollected,InstallCollected,EarlyCollected,InstallDueDate,TotalInstallsNumber
0,I1529359-0,2023,1,1,RETURN,M,70945935,7,D,0,...,1,0,0,0,0.0,0.0,0.0,1.0,2023-02-03,1
1,I1529363-0,2023,1,1,RETURN,B,70945940,7,D,0,...,1,0,0,0,0.0,0.0,0.0,1.0,2023-01-13,1
2,I1529367-0,2023,1,1,RETURN,B,71206276,1,D,0,...,0,0,0,0,0.0,0.0,1.0,0.0,2023-01-31,5
3,I1529367-0,2023,1,1,RETURN,B,71206276,1,D,0,...,0,0,0,0,0.0,0.0,0.0,0.0,2023-02-28,5
4,I1529367-0,2023,1,1,RETURN,B,71206276,1,D,0,...,0,1,0,1,0.0,0.0,0.0,0.0,2023-03-28,5


In [13]:
raw_df.columns

Index(['LoanID', 'AppYear', 'AppMonth', 'AppWeek', 'CustType', 'Frequency',
       'Application_ID', 'PortFolioID', 'LoanStatus', 'NewlyScored',
       'Accepted', 'Originated', 'OriginationDate', 'OriginatedAmount',
       'InstallmentNumber', 'PaidOffThisInstall', 'TotalRealizedPayin',
       'InstallStatusCode', 'InstallmentStatus', 'LoanPaidOff',
       'isLoanDefault', 'isRecentInstall', 'isInstallDefault',
       'ThirdPartyCollected', 'PartialCollected', 'InstallCollected',
       'EarlyCollected', 'InstallDueDate', 'TotalInstallsNumber'],
      dtype='object')

## XPD Feature Derivation (v6)

In [14]:
xpd_features_df = build_xpd_features_v6(raw_df)

raw_df_with_xpd = raw_df.merge(xpd_features_df, on='LoanID', how='left')

print('xpd_features_df shape:', xpd_features_df.shape)
print('raw_df_with_xpd shape:', raw_df_with_xpd.shape)
raw_df_with_xpd[['LoanID', 'InstallmentNumber', 'isInstallDefault', 'LoanPaidOff', 'fpd_flag', 'spd_flag', 'tpd_flag', 'xpd_count', 'earliest_default_inst', 'payoff_by_inst_5']].head(20)

xpd_features_df shape: (195561, 7)
raw_df_with_xpd shape: (862101, 35)


,LoanID,InstallmentNumber,isInstallDefault,LoanPaidOff,fpd_flag,spd_flag,tpd_flag,xpd_count,earliest_default_inst,payoff_by_inst_5
0,I1529359-0,1,0,1,0,0,0,0,0,1
1,I1529363-0,1,0,1,0,0,0,0,0,1
2,I1529367-0,2,0,0,0,0,0,1,5,0
3,I1529367-0,3,0,0,0,0,0,1,5,0
4,I1529367-0,5,1,0,0,0,0,1,5,0
5,I1529367-0,4,0,0,0,0,0,1,5,0
6,I1529367-0,1,0,0,0,0,0,1,5,0
7,I1529375-0,1,0,1,0,0,0,0,0,1
8,I1529382-0,3,1,0,0,0,1,2,3,0
9,I1529382-0,1,0,0,0,0,1,2,3,0


## XPD Validation Checks

In [6]:
loan_count_raw = raw_df['LoanID'].nunique()
loan_count_xpd = xpd_features_df['LoanID'].nunique()

if loan_count_raw != loan_count_xpd:
    raise AssertionError(f'Loan count mismatch: raw={loan_count_raw}, xpd={loan_count_xpd}')

binary_cols = ['fpd_flag', 'spd_flag', 'tpd_flag', 'payoff_by_inst_5']
for col in binary_cols:
    invalid_vals = sorted(set(xpd_features_df[col].dropna().unique()) - {0, 1})
    if invalid_vals:
        raise AssertionError(f'{col} contains non-binary values: {invalid_vals}')

if (xpd_features_df['xpd_count'] < (xpd_features_df['fpd_flag'] + xpd_features_df['spd_flag'] + xpd_features_df['tpd_flag'])).any():
    raise AssertionError('xpd_count must be >= fpd_flag + spd_flag + tpd_flag for every loan')

if (xpd_features_df['earliest_default_inst'] < 0).any():
    raise AssertionError('earliest_default_inst must be >= 0')

print('All XPD checks passed.')
print('Raw unique loans:', loan_count_raw)
print('XPD unique loans:', loan_count_xpd)

xpd_features_df.describe(include='all')

All XPD checks passed.
Raw unique loans: 195561
XPD unique loans: 195561


,LoanID,fpd_flag,spd_flag,tpd_flag,xpd_count,earliest_default_inst,payoff_by_inst_5
count,195561,195561.000000,195561.000000,195561.000000,195561.000000,195561.000000,195561.000000
unique,195561,NaN,NaN,NaN,NaN,NaN,NaN
top,I1529359-0,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,0.100905,0.087466,0.057384,0.328813,1.491218,0.304002
std,NaN,0.301203,0.282518,0.232575,0.480789,2.595734,0.459985
min,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,NaN,0.000000,0.000000,0.000000,1.000000,2.000000,1.000000


In [7]:
sample_check = raw_df_with_xpd[
    ['LoanID', 'InstallmentNumber', 'isInstallDefault', 'LoanPaidOff', 'fpd_flag', 'spd_flag', 'tpd_flag', 'xpd_count', 'earliest_default_inst', 'payoff_by_inst_5']
].sort_values(['LoanID', 'InstallmentNumber']).head(30)

sample_check

,LoanID,InstallmentNumber,isInstallDefault,LoanPaidOff,fpd_flag,spd_flag,tpd_flag,xpd_count,earliest_default_inst,payoff_by_inst_5
0,I1529359-0,1,0,1,0,0,0,0,0,1
17818,I1529362-0,1,0,0,0,0,0,0,0,1
17819,I1529362-0,2,0,1,0,0,0,0,0,1
1,I1529363-0,1,0,1,0,0,0,0,0,1
18190,I1529364-0,1,0,1,0,0,0,0,0,1
5954,I1529365-0,1,0,0,0,0,0,1,5,0
5953,I1529365-0,2,0,0,0,0,0,1,5,0
5955,I1529365-0,3,0,0,0,0,0,1,5,0
5956,I1529365-0,4,0,0,0,0,0,1,5,0
5957,I1529365-0,5,1,0,0,0,0,1,5,0


## Loan-Level Payin Features (v6)

In [17]:
loan_payin_df = build_loan_level_payin_v6(raw_df)

# Attach both XPD and loan-level payin features back to installment rows.
raw_df_enriched = (
    raw_df
    .merge(xpd_features_df, on='LoanID', how='left')
    .merge(loan_payin_df, on='LoanID', how='left')
)

print('loan_payin_df shape:', loan_payin_df.shape)
print('raw_df_enriched shape:', raw_df_enriched.shape)

loan_payin_df.head(20)

loan_payin_df shape: (195561, 7)
raw_df_enriched shape: (862101, 41)


,LoanID,OriginatedAmount,TotalRealizedPayin_sql,TotalRealizedPayin_from_install,payin_ratio_sql,payin_ratio_from_install,payin_ratio_gap
0,I1529359-0,1500.0,2109.68,2109.68,1.406453,1.406453,0.00
1,I1529362-0,600.0,763.57,763.57,1.272617,1.272617,0.00
2,I1529363-0,300.0,358.50,358.50,1.195000,1.195000,0.00
3,I1529364-0,500.0,700.00,700.00,1.400000,1.400000,0.00
4,I1529365-0,1500.0,2461.00,2461.00,1.640667,1.640667,0.00
5,I1529366-0,1500.0,1800.00,1800.00,1.200000,1.200000,0.00
6,I1529367-0,1500.0,4130.00,2525.00,2.753333,1.683333,-1.07
7,I1529369-0,1000.0,2157.16,1657.16,2.157160,1.657160,-0.50
8,I1529371-0,800.0,1696.00,1696.00,2.120000,2.120000,0.00
9,I1529372-0,500.0,1050.00,1050.00,2.100000,2.100000,0.00


In [16]:
# Quick reconciliation view: SQL total vs recomputed from installment payin.
recon_summary = {
    'loan_count': int(loan_payin_df['LoanID'].nunique()),
    'sum_sql_total': float(loan_payin_df['TotalRealizedPayin_sql'].sum()),
    'sum_install_total': float(loan_payin_df['TotalRealizedPayin_from_install'].sum()),
    'sum_ratio_gap': float(loan_payin_df['payin_ratio_gap'].sum()),
}

pd.DataFrame([recon_summary])

,loan_count,sum_sql_total,sum_install_total,sum_ratio_gap
0,195561,2.388575e+08,221470598.7,-28439.954214
